# Task 7: AI Agent Layer — Market Analysis Agent + Recommendation Agent

Extends the existing pipeline (data collection, knowledge repository, retrieval,
dashboard, Ollama/llama3.1:8b integration) with **explicit agent capabilities**,
per the clarified assignment brief:

- **Planning before execution**
- **Tool usage beyond the LLM itself**
- **Retrieval and use of evidence**
- **Analysis of risks, opportunities, and trends**
- **Autonomous decision-making**
- **Validation of recommendations before presenting them**

Workflow: **Goal → Plan → Retrieve → Analyze → Decide → Recommend → Validate**

Two specialized agents:
1. **MarketAnalysisAgent** — Plan → Retrieve → Analyze (produces risks/opportunities/trends)
2. **RecommendationAgent** — Decide → Recommend → Validate (produces final recommendations)


In [2]:
import json, os, requests, chromadb
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv("../.env")

chroma_client = chromadb.PersistentClient(path="../storage/chromadb")
collection = chroma_client.get_collection("lufthansa_intelligence")
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", token=os.getenv("HF_TOKEN"))

MODEL = "llama3.1:8b"  # unchanged — per brief, the focus is agent behaviour, not the model

print(f"Connected to knowledge repository: {collection.count()} documents")
print(f"LLM backend (unchanged): Ollama / {MODEL}")

/Users/namratabhoyar/Downloads/SRH_Master/NLP/lufthansa_intelligence/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5712.51it/s]


Connected to knowledge repository: 427 documents
LLM backend (unchanged): Ollama / llama3.1:8b


## Shared Tools

Both agents call these explicitly by name. This satisfies "tool usage beyond
the LLM itself" — the LLM is one tool among several, not the entire system.

In [3]:
def tool_search(query, n=8):
    """TOOL: semantic search over the knowledge repository (ChromaDB)."""
    emb = embedder.encode([query]).tolist()
    res = collection.query(query_embeddings=emb, n_results=n)
    return [{"title": m["title"], "text": t} for m, t in zip(res["metadatas"][0], res["documents"][0])]


def tool_llm(prompt, temperature=0.2):
    """TOOL: structured LLM call via Ollama's chat endpoint (uses the model's chat template)."""
    r = requests.post("http://localhost:11434/api/chat",
        json={"model": MODEL, "messages": [{"role": "user", "content": prompt}],
              "stream": False, "options": {"temperature": temperature}})
    return r.json()["message"]["content"]


def tool_validate(recommendations):
    """TOOL: rule-based validator — deterministic Python checks, not an LLM call."""
    valid_levels = {"High", "Medium", "Low"}
    issues = []
    for i, rec in enumerate(recommendations):
        evidence = rec.get("supporting_evidence", [])
        if len(evidence) != 3:
            issues.append(f"Rec {i+1}: expected 3 evidence items, found {len(evidence)}")
        for field in ["expected_impact", "risk_assessment"]:
            for cat, val in rec.get(field, {}).items():
                if val not in valid_levels:
                    issues.append(f"Rec {i+1}: {field}.{cat} = '{val}' is not High/Medium/Low")
    return {"total": len(recommendations), "passed": len(recommendations) - len(issues), "issues": issues}


def clean_json_array(text):
    """Helper: strip markdown fences / stray text the LLM sometimes adds around JSON."""
    text = text.replace("```json", "").replace("```", "")
    start, end = text.find("["), text.rfind("]") + 1
    try:
        return json.loads(text[start:end])
    except Exception:
        return []


print("Tools registered: tool_search, tool_llm, tool_validate")

Tools registered: tool_search, tool_llm, tool_validate


## Agent 1: MarketAnalysisAgent

Owns the **Plan → Retrieve → Analyze** stages. Produces structured
risks, opportunities, and trends with evidence.

In [4]:
class MarketAnalysisAgent:
    """Agent responsible for market intelligence: plans its search strategy,
    retrieves evidence, and analyzes it into risks/opportunities/trends."""

    def __init__(self, name="MarketAnalysisAgent"):
        self.name = name

    def plan(self, goal):
        print(f"\n[{self.name}] PLAN")
        plan_text = tool_llm(
            f"Goal: {goal}\n\nYou have a 'search' tool over a document repository and an 'llm' tool for analysis.\n"
            f"List your plan as 4 short numbered steps to gather and analyze market intelligence.",
            temperature=0.1
        )
        print(plan_text)
        return plan_text

    def retrieve(self):
        print(f"\n[{self.name}] RETRIEVE (tool: tool_search)")
        queries = {
            "risks": "Lufthansa risks competition financial decline strikes",
            "opportunities": "Lufthansa opportunities partnerships innovation expansion",
            "trends": "Lufthansa trends technology sustainability digital transformation"
        }
        retrieved = {}
        for category, query in queries.items():
            docs = tool_search(query)
            retrieved[category] = docs
            print(f"  tool_search('{query[:45]}...') -> {len(docs)} documents")
        return retrieved

    def analyze(self, retrieved):
        print(f"\n[{self.name}] ANALYZE (tool: tool_llm)")

        def extract(docs, kind, field):
            evidence_block = "\n".join(f"- {d['title']}: {d['text'][:250]}" for d in docs)
            prompt = f"""Extract the top 4 {kind}s for Lufthansa from these documents.

{evidence_block}

Respond ONLY with a valid JSON array:
[{{"title": "short title", "description": "2-3 sentence description", "evidence": "document title supporting this", "{field}": "High/Medium/Low", "confidence_score": "1-10"}}]"""
            return clean_json_array(tool_llm(prompt))

        risks = extract(retrieved["risks"], "risk", "severity")
        opportunities = extract(retrieved["opportunities"], "opportunity", "impact")
        trends = extract(retrieved["trends"], "trend", "impact")
        print(f"  Extracted: {len(risks)} risks, {len(opportunities)} opportunities, {len(trends)} trends")
        return {"risks": risks, "opportunities": opportunities, "trends": trends}

    def run(self, goal):
        plan = self.plan(goal)
        retrieved = self.retrieve()
        analysis = self.analyze(retrieved)
        return {"plan": plan, **analysis}


print("MarketAnalysisAgent defined.")

MarketAnalysisAgent defined.


## Agent 2: RecommendationAgent

Owns the **Decide → Recommend → Validate** stages. Takes the
MarketAnalysisAgent's output, synthesizes a CEO briefing, structures
evidence-based recommendations, validates them, and makes an autonomous
decision about whether to present them as final or flag them.

In [5]:
class RecommendationAgent:
    """Agent responsible for turning market analysis into validated,
    evidence-based strategic recommendations."""

    def __init__(self, name="RecommendationAgent"):
        self.name = name

    def plan(self, market_analysis):
        print(f"\n[{self.name}] PLAN")
        plan_text = tool_llm(
            f"You received {len(market_analysis['risks'])} risks, {len(market_analysis['opportunities'])} opportunities, "
            f"and {len(market_analysis['trends'])} trends from the MarketAnalysisAgent.\n"
            f"List your plan as 3 short numbered steps to decide on priorities, generate recommendations, and validate them.",
            temperature=0.1
        )
        print(plan_text)
        return plan_text

    def decide(self, market_analysis):
        print(f"\n[{self.name}] DECIDE (tool: tool_llm)")
        risks, opportunities, trends = market_analysis["risks"], market_analysis["opportunities"], market_analysis["trends"]

        summary = "RISKS:\n" + "\n".join(f"- {r.get('title')}: {r.get('description')}" for r in risks)
        summary += "\n\nOPPORTUNITIES:\n" + "\n".join(f"- {o.get('title')}: {o.get('description')}" for o in opportunities)
        summary += "\n\nTRENDS:\n" + "\n".join(f"- {t.get('title')}: {t.get('description')}" for t in trends)

        briefing = tool_llm(
            f"You are the AI Strategic Advisor to Lufthansa's CEO.\n{summary}\n\n"
            f"Write a 3-section executive briefing: WHAT HAPPENED, WHY IT MATTERS, "
            f"WHAT SHOULD MANAGEMENT DO NEXT (top 3 prioritized actions with a trade-off explanation).",
            temperature=0.3
        )
        print(briefing[:250] + "...")
        return briefing

    def recommend(self, briefing, market_analysis, feedback=None):
        print(f"\n[{self.name}] RECOMMEND (tool: tool_llm)")
        risks, opportunities, trends = market_analysis["risks"], market_analysis["opportunities"], market_analysis["trends"]

        evidence_pool = ""
        for items, label in [(risks, "RISKS"), (opportunities, "OPPORTUNITIES"), (trends, "TRENDS")]:
            evidence_pool += f"\n{label}:\n" + "".join(f"- {x.get('title')}: {x.get('evidence')}\n" for x in items)

        feedback_block = ""
        if feedback:
            feedback_block = "\nYour previous attempt had these problems, fix them this time:\n" + "\n".join(f"- {f}" for f in feedback) + "\n"

        prompt = f"""Based on this briefing and evidence pool, create exactly 3 evidence-based recommendations.

BRIEFING:
{briefing}

EVIDENCE POOL (each line is one valid evidence source you may cite):
{evidence_pool}
{feedback_block}
STRICT RULES — these will be checked programmatically:
- "supporting_evidence" must be a list of EXACTLY 3 strings, no more, no fewer
- each evidence string must be copied from a line in the evidence pool above
- "expected_impact" and "risk_assessment" values must be exactly one of: High, Medium, Low

Example of one correctly formatted recommendation:
{{"recommendation": "Expand digital partnerships to improve customer experience.",
  "supporting_evidence": ["Lufthansa Systems IT innovation", "Lufthansa Group IBM partnership", "Customer service reviews Trustpilot"],
  "expected_impact": {{"revenue_growth": "Medium", "market_differentiation": "High", "customer_acquisition": "High"}},
  "risk_assessment": {{"financial_risk": "Low", "operational_risk": "Medium", "strategic_risk": "Low"}}}}

Respond ONLY with a valid JSON array of exactly 3 such objects, no other text:"""
        recommendations = clean_json_array(tool_llm(prompt))
        print(f"  Generated {len(recommendations)} recommendations")
        return recommendations

    def validate(self, recommendations):
        print(f"\n[{self.name}] VALIDATE (tool: tool_validate)")
        report = tool_validate(recommendations)
        print(f"  Passed: {report['passed']}/{report['total']}")
        for issue in report["issues"]:
            print(f"    - {issue}")
        return report

    def decide_final_status(self, report):
        """Autonomous decision: present as final, or flag for review."""
        print(f"\n[{self.name}] AUTONOMOUS DECISION")
        if not report["issues"]:
            print("  All checks passed -> presenting recommendations as final")
            return "validated"
        else:
            print(f"  {len(report['issues'])} issue(s) found -> flagging for review, not silently auto-correcting")
            return "flagged_for_review"

    def run(self, market_analysis, max_retries=2):
        plan = self.plan(market_analysis)
        briefing = self.decide(market_analysis)

        feedback = None
        for attempt in range(max_retries + 1):
            recommendations = self.recommend(briefing, market_analysis, feedback=feedback)
            report = self.validate(recommendations)

            if not report["issues"]:
                break
            elif attempt < max_retries:
                print(f"\n[{self.name}] AUTONOMOUS DECISION")
                print(f"  {len(report['issues'])} issue(s) found -> retrying (attempt {attempt + 2}/{max_retries + 1}) with corrective feedback")
                feedback = report["issues"]

        status = self.decide_final_status(report)
        return {"plan": plan, "briefing": briefing, "recommendations": recommendations,
                "validation_report": report, "status": status, "attempts": attempt + 1}


print("RecommendationAgent defined.")

RecommendationAgent defined.


## Orchestrator — runs both agents in sequence

Goal → [MarketAnalysisAgent: Plan → Retrieve → Analyze] →
[RecommendationAgent: Decide → Recommend → Validate]

In [ ]:
GOAL = "If you were the CEO of Lufthansa today, what would you do next and why?"

print("="*60)
print("GOAL")
print("="*60)
print(GOAL)
#create one instance of each agent,__init__runs for each setting self name
market_agent = MarketAnalysisAgent()
recommendation_agent = RecommendationAgent()

print("\n" + "="*60)
print("STAGE 1 / MARKET ANALYSIS AGENT")
print("="*60)
#runs the agent internally(plan-retrieve-analyze),returns 4risk+4opp+4trends
market_result = market_agent.run(GOAL)

print("\n" + "="*60)
print("STAGE 2 / RECOMMENDATION AGENT")
print("="*60)
#runs agent 2,passing agent 1's output,plan-decide(ceo breifing)-reccomend-validate
recommendation_result = recommendation_agent.run(market_result)

print("\n" + "="*60)
print("AGENT RUN COMPLETE")
print("="*60)
print(f"Risks: {len(market_result['risks'])}, Opportunities: {len(market_result['opportunities'])}, Trends: {len(market_result['trends'])}")
print(f"Recommendations: {len(recommendation_result['recommendations'])}")
print(f"Validation: {recommendation_result['validation_report']['passed']}/{recommendation_result['validation_report']['total']} passed")
print(f"Recommend attempts: {recommendation_result['attempts']}")
print(f"Final status: {recommendation_result['status']}")

GOAL
If you were the CEO of Lufthansa today, what would you do next and why?

STAGE 1 / MARKET ANALYSIS AGENT

[MarketAnalysisAgent] PLAN
As the CEO of Lufthansa, my goal is to stay ahead of the competition and make informed decisions to drive growth and profitability. Here are the next 4 short steps I would take to gather and analyze market intelligence:

**Step 1: Search for Industry Trends and Competitor Analysis (Document Repository)**

* Use the search tool to scan industry reports, news articles, and research papers on topics such as:
	+ Air travel demand trends
	+ Competition from low-cost carriers and new entrants
	+ Emerging technologies like electric or hydrogen-powered aircraft
	+ Regulatory changes affecting the airline industry
* Identify key findings and insights that can inform our business strategy

**Step 2: Analyze Market Data and Customer Behavior (LLM Tool)**

* Use the LLM tool to analyze large datasets on:
	+ Passenger demographics and travel patterns
	+ Route per

## Save outputs (the existing dashboard `app.py` reads these — no changes needed there)

In [7]:
os.makedirs("../data", exist_ok=True)

with open("../data/intelligence_results.json", "w", encoding="utf-8") as f:
    json.dump({
        "risks": market_result["risks"],
        "opportunities": market_result["opportunities"],
        "trends": market_result["trends"]
    }, f, indent=2, ensure_ascii=False)

with open("../data/ceo_recommendation.txt", "w", encoding="utf-8") as f:
    f.write(recommendation_result["briefing"])

with open("../data/evidence_based_recommendations.json", "w", encoding="utf-8") as f:
    json.dump({"recommendations": recommendation_result["recommendations"]}, f, indent=2, ensure_ascii=False)

with open("../data/agent_log.json", "w", encoding="utf-8") as f:
    json.dump({
        "goal": GOAL,
        "market_agent_plan": market_result["plan"],
        "recommendation_agent_plan": recommendation_result["plan"],
        "validation_report": recommendation_result["validation_report"],
        "recommend_attempts": recommendation_result["attempts"],
        "final_status": recommendation_result["status"]
    }, f, indent=2, ensure_ascii=False)

print("Saved all outputs. The Streamlit dashboard (app.py) will reflect these on refresh — no dashboard changes needed.")

Saved all outputs. The Streamlit dashboard (app.py) will reflect these on refresh — no dashboard changes needed.
